# **Setup**

- https://www.kdnuggets.com/turboquant-is-the-compression-and-performance-worth-the-hype 
- https://pypi.org/project/turboquant/

In [1]:
!pip install turboquant -q

In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant import TurboQuantCache

# **Load LLM Model**

In [3]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# **Prompt**

In [4]:
prompt = "Explain the history of the universe in great detail. " * 20 
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# **Custom Function**

In [5]:
def run_unified_benchmark(use_tq=False):
    torch.cuda.empty_cache()
    
    # Initializing the specific cache type
    cache = TurboQuantCache(bits=3) if use_tq else None
    
    start_time = time.time()
    with torch.no_grad():
        # Running the model to generate output tokens
        outputs = model.generate(**inputs, max_new_tokens=100, past_key_values=cache)
    
    duration = time.time() - start_time
    
    # Isolating the Cache Memory
    # Instead of measuring the whole 2GB model, we measure the generated Cache size
    # For a 1.1B model: [Layers: 22, Heads: 32, Head_Dim: 64]
    num_tokens = outputs.shape[1]
    elements = 22 * 32 * 64 * num_tokens * 2 # Key + Value
    
    if use_tq:
        mem_mb = (elements * 3) / (8 * 1024 * 1024) # 3-bit calculation
    else:
        mem_mb = (elements * 16) / (8 * 1024 * 1024) # 16-bit calculation
        
    return duration, mem_mb

# **TQ vs Non-TQ Comparison**

In [6]:
base_time, base_mem = run_unified_benchmark(use_tq=False)
tq_time, tq_mem = run_unified_benchmark(use_tq=True)

print(f"--- THE VERDICT ---")
print(f"Baseline (FP16) Cache: {base_mem:.2f} MB")
print(f"TurboQuant (3-bit) Cache: {tq_mem:.2f} MB")
print(f"Speedup: {base_time / tq_time:.2f}x")
print(f"Memory Saved: {base_mem - tq_mem:.2f} MB")

--- THE VERDICT ---
Baseline (FP16) Cache: 42.45 MB
TurboQuant (3-bit) Cache: 7.86 MB
Speedup: 0.99x
Memory Saved: 34.59 MB
